<div style="display: flex; justify-content: flex-start; align-items: center;">
   <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20252/Module%204%20-%20Testing%20the%20Transformers%20Library/1-Testing%20Transformers%20Library_Python.ipynb" target="_parent"><img 
   src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</div>

# Testing the Transformers Library

This notebook covers two major capabilities of the Hugging Face `transformers` library:

**Part 1 — Contextual Word Embeddings with BERT**  
Traditional word embedding models (Word2Vec, GloVe) give every word a single fixed vector.  
BERT produces *context-aware* embeddings: the same word gets a different representation depending on the surrounding sentence. We visualise this and confirm it with cosine similarity scores.

**Part 2 — The Pipeline API**  
`pipeline()` is the fastest path from a task name to a working model — it handles tokenisation, inference, and output formatting in a single call. We cover six tasks:
1. Sentiment Analysis
2. Named Entity Recognition (NER)
3. Question Answering
4. Text Generation
5. Summarisation
6. Translation (Arabic ↔ English)

---

## Part 1: Context-Aware Word Embeddings with BERT

Traditional word embeddings (Word2Vec, GloVe) assign a **fixed** vector to each word regardless of how it is used. The word "bank" always gets the same representation, whether it means a financial institution or a riverbank.

**BERT** (Bidirectional Encoder Representations from Transformers) produces **contextual** embeddings. Before generating a vector for any token, BERT reads the *entire sentence* bidirectionally — so the vector for "bank" changes depending on the surrounding words.

Every token is represented as a **768-dimensional vector** (in BERT-base). Words used in similar contexts will have similar vectors — we can measure this using **cosine similarity**.

In [ ]:
# transformers — pre-trained models, tokenizers, and the Pipeline API
# torch       — PyTorch deep learning framework (required backend for transformers)
%pip install transformers torch

### Inspecting BERT Token Embeddings

We feed the sentence *"He went to the bank to deposit money."* to BERT and print the embedding vector for each token.

**Things to notice in the output:**
- The tokenizer automatically adds `[CLS]` at the start and `[SEP]` at the end — these are special control tokens BERT was trained with
- `[CLS]` (classification token) accumulates a summary of the whole sentence across all layers
- Each word gets a unique 768-dimensional vector shaped by its context in this specific sentence
- We print only the first 5–10 values — all 768 are used in practice

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model.
# "bert-base-uncased" is the standard 12-layer BERT, trained on lowercased English.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()  # disable dropout — we are running inference, not training

sentence = "He went to the bank to deposit money."

# Tokenise the sentence → returns a dict with input_ids and attention_mask tensors
# return_tensors='pt' means PyTorch tensors
inputs = tokenizer(sentence, return_tensors='pt')

# Convert token IDs back to human-readable tokens so we can print them
# e.g. [101, 2002, 2253, ...] → ['[CLS]', 'he', 'went', ...]
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Run the model — torch.no_grad() disables gradient tracking (saves memory, not training)
with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state: shape (batch=1, seq_len, hidden=768) — one 768-dim vector per token
# squeeze(0) removes the batch dimension → shape (seq_len, 768)
embeddings = outputs.last_hidden_state.squeeze(0)

print(f"Sentence : {sentence}")
print(f"Tokens   : {tokens}\n")
print(f"{'Token':<12} First 5 embedding values")
print("-" * 55)
for token, emb in zip(tokens, embeddings):
    print(f"  {token:<12} {emb[:5].numpy().round(4)}")

### Comparing Embeddings Across Contexts: Cosine Similarity

**Cosine similarity** measures the angle between two vectors:
- **1.0** — vectors point in exactly the same direction (nearly identical meaning in context)
- **~0.5** — moderate similarity
- **0.0** — completely unrelated

**The experiment:**

| Comparison | Expected result | Why |
|---|---|---|
| "apple" (fruit) vs "orange" | High similarity | Both fruits in similar sentences |
| "apple" (company) vs "Microsoft" | High similarity | Both tech companies in similar sentences |
| "apple" (fruit) vs "apple" (company) | Lower similarity | Same spelling, very different contexts |

If BERT truly produces contextual embeddings, the last pair should score *lower* than the first two — meaning BERT distinguishes the two meanings of "apple".

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

def get_token_embedding(sentence, target_word):
    """Return the contextual embedding vector for target_word in the given sentence.

    Handles subword tokenisation: if the word is split into multiple tokens
    (e.g. "playing" → ["play", "##ing"]), their embeddings are averaged into one vector.
    """
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    embeddings = outputs.last_hidden_state.squeeze(0)  # shape: (seq_len, 768)

    # Find where BERT's subword tokens for the target word appear in the token list
    target_tokens = tokenizer.tokenize(target_word)
    matches = []
    for i in range(len(tokens) - len(target_tokens) + 1):
        if tokens[i:i + len(target_tokens)] == target_tokens:
            matches = list(range(i, i + len(target_tokens)))
            break

    if not matches:
        raise ValueError(f"'{target_word}' not found in tokens: {tokens}")

    # Mean-pool the subword embeddings → single vector representing the whole word
    return embeddings[matches].mean(dim=0)

# Same word "apple" used in two very different contexts
sentence_fruit     = "He ate a fresh apple and enjoyed the fruit."
sentence_company   = "Apple released a new product in the computer market."
sentence_orange    = "An orange is a juicy fruit."
sentence_microsoft = "Microsoft computer was running the latest software."

# Extract context-dependent embeddings
apple_fruit   = get_token_embedding(sentence_fruit,    "apple")
apple_company = get_token_embedding(sentence_company,  "apple")
orange        = get_token_embedding(sentence_orange,   "orange")
microsoft     = get_token_embedding(sentence_microsoft, "Microsoft")

# Cosine similarity: 1.0 = identical direction, 0.0 = orthogonal (unrelated)
sim_fruit_orange    = F.cosine_similarity(apple_fruit,   orange,        dim=0)
sim_company_ms      = F.cosine_similarity(apple_company, microsoft,     dim=0)
sim_apple_vs_apple  = F.cosine_similarity(apple_fruit,   apple_company, dim=0)

print("Cosine Similarity Results")
print("=" * 55)
print(f"'apple' (fruit)   vs 'orange'         : {sim_fruit_orange.item():.4f}  ← both fruits")
print(f"'apple' (company) vs 'Microsoft'       : {sim_company_ms.item():.4f}  ← both tech companies")
print(f"'apple' (fruit)   vs 'apple' (company) : {sim_apple_vs_apple.item():.4f}  ← same word, different meaning")
print()
print("Interpretation: 'apple' is closer to 'orange' when used as a fruit,")
print("and closer to 'Microsoft' when used as a tech brand — BERT captures this distinction.")

---

---

## Part 2: The Transformers Pipeline API

The `pipeline()` function is the simplest way to run a pre-trained model:

```python
from transformers import pipeline
pipe = pipeline("task-name")          # downloads model automatically
result = pipe("your input text")      # tokenise → infer → format result
```

It wraps tokenisation, model inference, and result post-processing into a single call.  
We demonstrate six common NLP tasks below.

### Task 1: Sentiment Analysis (Text Classification)

Sentiment analysis classifies text as positive or negative (or other labels depending on the model).

**How it works internally:**
1. The text is tokenised into subword token IDs
2. A fine-tuned BERT-family model produces a score for each class
3. The class with the highest score is returned as the predicted label

When no `model=` argument is given, Hugging Face selects a default model  
(`distilbert-base-uncased-finetuned-sst-2-english` in this case).  
In production, always pin a model explicitly — defaults can change between library versions.

In [ ]:
from transformers import pipeline

# "sentiment-analysis" selects a default fine-tuned DistilBERT model automatically.
# Always specify model= in production to avoid behaviour changes when defaults update.
classifier = pipeline("sentiment-analysis")

# Single input → list with one result dict: [{'label': ..., 'score': ...}]
result = classifier("I love using Hugging Face!")
print("Single input:", result)

# Batch input — more efficient than calling the pipeline one string at a time
texts = [
    "I hate this product",
    "This is amazing!",
    "It's okay, nothing special"
]

print("\nBatch input:")
results = classifier(texts)
for text, res in zip(texts, results):
    print(f"  {res['label']:8s} ({res['score']:.4f})  ←  \"{text}\"")

### Task 2: Named Entity Recognition (NER)

NER identifies spans of text that refer to real-world entities and classifies them by type:

| Label | Type | Examples |
|---|---|---|
| `PER` | Person | "John", "Marie Curie" |
| `LOC` | Location | "New York", "Jordan" |
| `ORG` | Organisation | "Google", "United Nations" |
| `MISC` | Miscellaneous | "English", "World Cup" |

**`aggregation_strategy="simple"`** merges consecutive subword tokens belonging to the same entity into one entry — e.g. "New" + "York" → `"New York"` with label `LOC`.

In [ ]:
# aggregation_strategy="simple" merges consecutive tokens for the same entity
# e.g. "New" + "York" → single entry "New York" with label LOC
ner = pipeline("ner", aggregation_strategy="simple")

text = "My name is John and I live in New York. I work at Google."
entities = ner(text)

print(f"Input: {text}\n")
print(f"{'Entity':<15} {'Label':<8} {'Score':<8} Span")
print("-" * 50)
for entity in entities:
    span = f"[{entity['start']}:{entity['end']}]"
    print(f"  {entity['word']:<13} {entity['entity_group']:<8} {entity['score']:.4f}   {span}")

### Task 3: Extractive Question Answering

Given a **context paragraph** and a **question**, the model finds the exact span of text within the context that best answers the question — it does not generate new words, it *extracts* the answer.

**How it works:**
1. Input is formatted as: `[CLS] question [SEP] context [SEP]`
2. The model outputs two probability distributions over the token positions: one for the answer **start**, one for the answer **end**
3. The tokens between the highest-scoring start and end positions form the extracted answer

We use `deepset/bert-base-cased-squad2`, fine-tuned on the SQuAD 2.0 reading comprehension dataset.

> **Note (transformers 5.x):** `pipeline("question-answering")` was removed in transformers 5.0. We use `AutoModelForQuestionAnswering` directly instead, which is more educational as it shows the underlying steps.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# deepset/bert-base-cased-squad2 is fine-tuned on SQuAD 2.0 — a large reading
# comprehension dataset where some questions intentionally have no answer in the context.
tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
model = AutoModelForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
model.eval()

def answer_question(question, context):
    """Extract the answer span from the context paragraph.

    The model outputs start_logits and end_logits — one score per token
    for where the answer begins and ends in the context.
    """
    # Input format: [CLS] question [SEP] context [SEP]
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # argmax selects the token position with the highest score
    start = outputs.start_logits.argmax()
    end   = outputs.end_logits.argmax() + 1  # +1 because Python slicing end is exclusive

    # Decode the selected token range back to a readable string
    answer_tokens = inputs["input_ids"][0][start:end]
    return tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(answer_tokens))

context = """
Hugging Face is a company that develops tools for building applications using machine learning.
They are especially known for their work in natural language processing. The company was founded in 2016
and is headquartered in New York.
"""

questions = [
    "When was Hugging Face founded?",
    "Where is Hugging Face headquartered?",
    "What is Hugging Face known for?"
]

for question in questions:
    answer = answer_question(question, context)
    print(f"Q: {question}")
    print(f"A: {answer}\n")

### Task 4: Text Generation

Text generation uses a **decoder-only** language model (GPT-2 here) to continue a given prompt.  
The model predicts the next token one step at a time by sampling from the probability distribution over the vocabulary.

**Key generation parameters:**

| Parameter | Effect |
|---|---|
| `max_length` | Total token limit including the prompt |
| `num_return_sequences` | How many independent completions to generate |
| `temperature` | Randomness control — lower (e.g. 0.3) = more predictable, higher (e.g. 1.5) = more creative |
| `do_sample=True` | Sample from the distribution; `False` would use greedy decoding (always the highest-probability token) |

In [ ]:
# GPT-2 is a decoder-only transformer — it generates text left-to-right
# by repeatedly predicting the next most likely token.
generator = pipeline("text-generation", model="gpt2")

prompts = [
    "The future of artificial intelligence is",
    "In a world where robots exist,"
]

for prompt in prompts:
    generated = generator(
        prompt,
        max_length=50,             # total tokens in output (includes the prompt itself)
        num_return_sequences=2,    # produce 2 independent completions for comparison
        temperature=0.7,           # < 1.0 makes output more focused; > 1.0 more random
        do_sample=True,            # sample from the distribution (not greedy/deterministic)
        pad_token_id=generator.tokenizer.eos_token_id  # suppress padding warnings
    )

    print(f"Prompt: {prompt}")
    for i, gen in enumerate(generated):
        print(f"  [{i+1}] {gen['generated_text']}")
    print()

### Task 5: Abstractive Text Summarisation

Summarisation condenses a long text into a shorter version. We use `facebook/bart-large-cnn` — a BART model fine-tuned on the CNN/DailyMail news dataset.

**Abstractive vs Extractive:**
- **Extractive** — selects and joins existing sentences from the source
- **Abstractive** — generates *new* sentences that may not appear verbatim in the source (what BART does)

**Generation parameters:**

| Parameter | Effect |
|---|---|
| `max_length` | Upper bound on summary length in tokens |
| `min_length` | Lower bound — prevents trivially short output |
| `length_penalty` | Values > 1.0 encourage longer sequences |
| `num_beams` | Beam search width — higher = better quality but slower |

> **Note (transformers 5.x):** `pipeline("summarization")` was removed in transformers 5.0. We use `AutoModelForSeq2SeqLM` with `model.generate()` directly instead.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# facebook/bart-large-cnn: BART encoder-decoder fine-tuned on CNN/DailyMail news articles.
# Encoder reads the full article; decoder generates an abstractive summary.
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

article = """
Machine learning is a subset of artificial intelligence that enables computers to learn and improve
from experience without being explicitly programmed. It focuses on the development of computer programs
that can access data and use it to learn for themselves. The process of learning begins with observations
or data, such as examples, direct experience, or instruction, in order to look for patterns in data and
make better decisions in the future based on the examples that we provide. The primary aim is to allow
the computers to learn automatically without human intervention or assistance and adjust actions accordingly.
"""

# Tokenise the article; max_length=1024 is BART's maximum context window
inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)

# model.generate() runs beam search to find a high-quality summary sequence
with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=50,       # upper bound on summary length (in tokens)
        min_length=25,       # prevent very short output
        length_penalty=2.0,  # penalise short sequences; > 1.0 encourages longer summaries
        num_beams=4          # explore 4 candidate sequences simultaneously (beam search)
    )

# Decode token IDs back to text; skip_special_tokens removes [BOS], [EOS], etc.
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Original : {len(article.split())} words")
print(f"Summary  : {len(summary.split())} words\n")
print("Summary:")
print(summary)

### Task 6: Translation — Arabic ↔ English

Translation converts text from one language to another using a **sequence-to-sequence** model: an encoder reads the source language, and a decoder generates the target language.

We use models from the **Helsinki-NLP** collection, which provides pre-trained translation models for hundreds of language pairs via the Marian NMT framework:
- **Arabic → English:** `Helsinki-NLP/opus-mt-ar-en`
- **English → Arabic:** `Helsinki-NLP/opus-mt-en-ar`

> **Note (transformers 5.x):** `AutoTokenizer` does not support `MarianConfig` in transformers 5.x. Use `MarianTokenizer` and `MarianMTModel` directly instead of the `Auto` classes.

In [ ]:
from transformers import MarianTokenizer, MarianMTModel
import torch

# In transformers 5.x, AutoTokenizer does not support MarianConfig.
# Use MarianTokenizer and MarianMTModel directly — this is the correct approach.

# ── Arabic → English ──────────────────────────────────────────────────────────
ar_en_name      = "Helsinki-NLP/opus-mt-ar-en"
ar_en_tokenizer = MarianTokenizer.from_pretrained(ar_en_name)
ar_en_model     = MarianMTModel.from_pretrained(ar_en_name)
ar_en_model.eval()

def translate_ar_en(text):
    """Translate a single Arabic string to English."""
    inputs = ar_en_tokenizer(text, return_tensors="pt", padding=True)
    with torch.no_grad():
        output_ids = ar_en_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
    return ar_en_tokenizer.decode(output_ids[0], skip_special_tokens=True)

arabic_texts = [
    "مرحبا، كيف حالك اليوم؟",
    "تعلم الآلة مجال رائع ومتطور.",
    "أود أن أطلب فنجان قهوة من فضلك."
]

print("Arabic → English")
print("-" * 50)
for text in arabic_texts:
    print(f"  AR: {text}")
    print(f"  EN: {translate_ar_en(text)}\n")

# ── English → Arabic ──────────────────────────────────────────────────────────
en_ar_name      = "Helsinki-NLP/opus-mt-en-ar"
en_ar_tokenizer = MarianTokenizer.from_pretrained(en_ar_name)
en_ar_model     = MarianMTModel.from_pretrained(en_ar_name)
en_ar_model.eval()

def translate_en_ar(text):
    """Translate a single English string to Arabic."""
    inputs = en_ar_tokenizer(text, return_tensors="pt", padding=True)
    with torch.no_grad():
        output_ids = en_ar_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
    return en_ar_tokenizer.decode(output_ids[0], skip_special_tokens=True)

english_texts = [
    "Artificial intelligence is transforming every industry.",
    "The weather is beautiful today.",
    "Please send me the report by tomorrow."
]

print("English → Arabic")
print("-" * 50)
for text in english_texts:
    print(f"  EN: {text}")
    print(f"  AR: {translate_en_ar(text)}\n")